# Prompt eval

## 1. generate test cases

In [1]:
import json
#load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
    {
        "task": "Description of task",
    },
    ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [5]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with keys 'bucket' and 'key'"}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"}, {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters)'}]


In [6]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

## 2. run test eval

In [18]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [19]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [20]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [21]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [22]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

KeyError: 'score'

In [ ]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 URI Parser\n\nHere's a comprehensive solution with multiple approaches:\n\n## Simple Solution\n\n```python\ndef parse_s3_uri(uri):\n    \"\"\"\n    Parse an AWS S3 bucket URI and return bucket and key.\n    \n    Args:\n        uri (str): S3 URI in format 's3://bucket-name/path/to/file'\n    \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' keys\n    \n    Raises:\n        ValueError: If URI format is invalid\n    \"\"\"\n    if not uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = uri[5:]\n    \n    # Split on first '/'\n    parts = uri_without_prefix.split('/', 1)\n    \n    if len(parts) != 2:\n        raise ValueError(\"URI must contain both bucket and key (e.g., 's3://bucket/key')\")\n    \n    bucket, key = parts\n    \n    if not bucket:\n        raise ValueError(\"Bucket name cannot be empty\")\n    \n    return {'bucket': bucket, 'key': ke